# 02 — Prétraitement, baseline et pré-entraînement (Phases 2 à 4)
Les données préparées viennent de `python scripts/prepare_data.py`. Ce notebook **vérifie** et **visualise** ; l'entraînement se lance via les scripts.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from ddos_tl.config import load_config, resolve

CONFIG = "configs/default.yaml"   # utiliser "configs/synthetic.yaml" pour un test rapide
cfg = load_config(CONFIG)
LABEL = cfg["data"]["label_column"]

In [2]:
from ddos_tl.data.pipeline import load_splits
processed = resolve(cfg["paths"]["processed"])
tgt = load_splits(processed, "target")
print("Cible  :", {k: tgt[f"X_{k}"].shape for k in ("train", "val", "test")}, tgt["class_names"])
try:
    src = load_splits(processed, "source")
    print("Source :", {k: src[f"X_{k}"].shape for k in ("train", "val", "test")}, src["class_names"])
    print("Mêmes features source/cible :", src["feature_columns"] == tgt["feature_columns"])
except FileNotFoundError as e:
    print(e)

FileNotFoundError: C:\Users\sayda\Desktop\ddos-transfer-learning\ddos-transfer-learning\data\processed\target_splits.npz introuvable : lance d'abord scripts/prepare_data.py

## Contrôle anti-fuite : les statistiques de normalisation viennent du train uniquement
Le train doit avoir une moyenne ≈ 0 et un écart-type ≈ 1 ; val et test légèrement différents (c'est normal).

In [ ]:
for part in ("train", "val", "test"):
    X = tgt[f"X_{part}"]
    print(f"{part:5s} mean={X.mean():+.3f} std={X.std():.3f} min={X.min():+.1f} max={X.max():+.1f}")

In [ ]:
pd.DataFrame(tgt["counts"]).fillna(0).astype(int)   # classes par split

## Lancer l'entraînement (dans un terminal, à la racine du projet)
```bash
python scripts/pretrain.py                           # Phase 4 : backbone pré-entraîné sur la source
python scripts/run_transfer.py --strategy scratch    # Phase 3 : baseline DNN
```

In [ ]:
meta_path = resolve(cfg["paths"]["models"]) / "pretrained_meta.json"
if meta_path.exists():
    from ddos_tl.utils.io import load_json
    m = load_json(meta_path)
    print({k: m[k] for k in ("source_dataset", "created_utc", "n_features", "source_classes", "source_test_macro_f1")})
else:
    print("Pas encore de pré-entraînement : lance scripts/pretrain.py")